#### Libraries

In [1]:
import pandas as pd
import os
import re
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.dates as mdates
import numpy as np
import matplotlib.patches as mpatches


In [28]:
# === Apply or skip filtering based on toggle ===
apply_filter = False   # 🔘 Set to True to use the Excel filter(Model output), False to include ALL chargers

#### Outlet-Level Thermal Analysis

**Goal:** Build an outlet-level, day-granular timeline across CTD, CLG, PLE, and Session logs to study thermal anomalies and PLE/ple spikes.  

#### 1) Dataset schema audit
**Goal:** Inspect columns, dtypes and a few rows.  

In [2]:
# 📁 Define paths
DATA_DIR = os.path.join("..", "data_all")

In [3]:
ctd_df = pd.read_csv(os.path.join(DATA_DIR, "CTD_last_year.csv"))
# clg_df = pd.read_csv(os.path.join(DATA_DIR, "CLG_total.xlsx"))
ple_df = pd.read_csv(os.path.join(DATA_DIR, "PLE_last_year.csv"))
sess_df = pd.read_csv(os.path.join(DATA_DIR, "SeccSessionStop_last_year.csv"))


# 🔍 Function to summarize
def summarize_dataset(name, df):
    print(f"\n=== 🔎 {name} Dataset ===")
    print("🧾 Columns:", df.columns.tolist())
    print("📊 Dtypes:")
    print(df.dtypes)
    print("👀 Head:")
    display(df.head(5))

# 📥 Load and summarize each
for name, df in [
    ("CTD", ctd_df),
    # ("CLG", clg_df),
    ("PLE", ple_df),
    ("SeccSessionStop", sess_df)
]:
    summarize_dataset(name, df)


C:\Users\z0054bay\AppData\Local\Temp\ipykernel_11040\2530189687.py:4: DtypeWarning: Columns (3,7) have mixed types. Specify dtype option on import or set low_memory=False.
  sess_df = pd.read_csv(os.path.join(DATA_DIR, "SeccSessionStop_last_year.csv"))



=== 🔎 CTD Dataset ===
🧾 Columns: ['@timestamp', 'IDOutlet', 'cause', 'cableid', 'nowcur', 'Contacttemp', 'Contacttemp2', 'diff', 'factor', 'EVID', 'Energy', 'duration', '@ptr']
📊 Dtypes:
@timestamp       object
IDOutlet         object
cause            object
cableid          object
nowcur            int64
Contacttemp     float64
Contacttemp2    float64
diff            float64
factor          float64
EVID             object
Energy          float64
duration        float64
@ptr             object
dtype: object
👀 Head:


,@timestamp,IDOutlet,cause,cableid,nowcur,Contacttemp,Contacttemp2,diff,factor,EVID,Energy,duration,@ptr
0,2025-03-27 13:34:29.067000+00:00,yk59l51,debug,DC1,199,45.4,40.4,5.0,0.2281,EV-T2M4CC-DC,9.9,6.6667,CmwKKQoVNjg5MTE3MzE0NzAwOm1lc3NhZ2VzEAAiDgiPnr...
1,2025-03-27 13:35:04.393000+00:00,0WO7nc1,debug,DC1,279,34.8,29.8,5.0,0.1247,EV-T2M4CC-DC,6.2,3.5462,CmwKKQoVNjg5MTE3MzE0NzAwOm1lc3NhZ2VzEAAiDgiPnr...
2,2025-03-27 13:35:06.841000+00:00,HfJX291,debug,DC1,288,63.4,53.4,10.0,0.2201,EV-T2M4CC-DC,16.1,9.1046,CmwKKQoVNjg5MTE3MzE0NzAwOm1lc3NhZ2VzEAAiDgiPnr...
3,2025-03-27 13:35:46.436000+00:00,NWYmy61,debug,DC1,390,53.6,43.5,10.1,0.1374,EV-T2M4CC-DC,9.2,3.6340,CmwKKQoVNjg5MTE3MzE0NzAwOm1lc3NhZ2VzEAAiDgiPnr...
4,2025-03-27 13:36:50.288000+00:00,mrPnq12,debug,DC2,391,59.7,54.7,5.0,0.1527,EV-T2M4CC-DC,9.1,4.6232,CmwKKQoVNjg5MTE3MzE0NzAwOm1lc3NhZ2VzEAAiDgiPnr...



=== 🔎 PLE Dataset ===
🧾 Columns: ['IDOutlet', '@timestamp', 'Error', '@ptr']
📊 Dtypes:
IDOutlet      object
@timestamp    object
Error         object
@ptr          object
dtype: object
👀 Head:


,IDOutlet,@timestamp,Error,@ptr
0,bM0EQk2,2025-03-27 13:44:43.636000+00:00,Error,CmsKKQoVNjg5MTE3MzE0NzAwOm1lc3NhZ2VzEAAiDgiPnr...
1,bM0EQk2,2025-03-27 13:44:59.619000+00:00,Error,CmwKKQoVNjg5MTE3MzE0NzAwOm1lc3NhZ2VzEAYiDgiPnr...
2,x4Uawq2,2025-03-27 13:49:24.886000+00:00,Error,CmwKKQoVNjg5MTE3MzE0NzAwOm1lc3NhZ2VzEAciDgiPnr...
3,x4Uawq2,2025-03-27 13:49:32.905000+00:00,Error,CmsKKQoVNjg5MTE3MzE0NzAwOm1lc3NhZ2VzEAYiDgiPnr...
4,jERN9L1,2025-03-27 13:58:29.177000+00:00,Error,CmwKKQoVNjg5MTE3MzE0NzAwOm1lc3NhZ2VzEAAiDgiPnr...



=== 🔎 SeccSessionStop Dataset ===
🧾 Columns: ['IDOutlet', '@timestamp', 'pt1000_1_T', 'pt1000_2_T', 'duration', 'energy', 'diff', 'Energydiff', '@ptr']
📊 Dtypes:
IDOutlet       object
@timestamp     object
pt1000_1_T    float64
pt1000_2_T     object
duration      float64
energy        float64
diff          float64
Energydiff     object
@ptr           object
dtype: object
👀 Head:


,IDOutlet,@timestamp,pt1000_1_T,pt1000_2_T,duration,energy,diff,Energydiff,@ptr
0,Ubbd7a2,2025-03-27 13:34:04.271000+00:00,42.7572,40.425,1239.0,23389.0,2.3322,0.09971,CmwKKQoVNjg5MTE3MzE0NzAwOm1lc3NhZ2VzEAAiDgiPnr...
1,bPeYpZ2,2025-03-27 13:34:12.107000+00:00,36.7971,35.5014,2726.0,53660.0,1.2957,0.02415,CmwKKQoVNjg5MTE3MzE0NzAwOm1lc3NhZ2VzEAAiDgiPnr...
2,64n1W82,2025-03-27 13:34:15.901000+00:00,21.5081,22.0264,5664.0,223247.0,0.5183,0.002322,CmwKKQoVNjg5MTE3MzE0NzAwOm1lc3NhZ2VzEAAiDgiPnr...
3,FDL7ZG2,2025-03-27 13:34:21.755000+00:00,28.2457,29.8005,2919.0,46516.0,1.5548,0.03343,CmwKKQoVNjg5MTE3MzE0NzAwOm1lc3NhZ2VzEAAiDgiPnr...
4,U9Dn7E1,2025-03-27 13:34:23.557000+00:00,28.2457,27.4682,1802.0,40557.0,0.7774,0.01917,CmwKKQoVNjg5MTE3MzE0NzAwOm1lc3NhZ2VzEAEiDgiPnr...


#### 2) PLE: Extract outlet from @message
**Purpose:** Parse outlet id (DC1/DC2/…) from PLE logs.  

In [4]:
# # 📁 Load dataset
# DATA_DIR = os.path.join("..", "data_all")
# ple_path = os.path.join(DATA_DIR, "PLE.csv")
# ple_df = pd.read_csv(ple_path)

# # ✅ Step 1: Parse outlet info from `@message`
# ple_df["outlet"] = ple_df["@ptr"].str.extract(r"(DC\d+)_CableTempSensor")

# # 🔍 Optional check: Print unique outlets
# print("✅ Extracted outlets:", ple_df["outlet"].dropna().unique())

# # ✅ Step 2: Save the updated file back
# ple_df.to_csv(ple_path, index=False)
# print("✅ PLE_timeseries.xlsx updated with 'outlet' column.")


- 3) Normalize CTD outlets
**Purpose:** Extract integer outlet id from `cableid`.  
- 4) Normalize CLG outlets
**Purpose:** Extract integer outlet id from `cableid`.
- 5) Normalize PLE outlets
**Purpose:** Ensure integer `outlet` in PLE.  
- 6) Normalize Session outlets
**Purpose:** Strip quotes and parse `outlet` to integer.  

In [5]:
# === Step 1: Split IDOutlet into @logStream and outlet ===

def split_idoutlet(id_str):
    """
    Split IDOutlet into charger (@logStream) and outlet (last digit).
    """
    if pd.isna(id_str):
        return (pd.NA, pd.NA)
    s = str(id_str)
    if s[-1].isdigit():
        return (s[:-1], int(s[-1]))
    else:
        return (s, pd.NA)

# --- Apply to CTD dataset ---
ctd_split = ctd_df["IDOutlet"].apply(split_idoutlet)
ctd_df["@logStream"] = ctd_split.apply(lambda x: x[0])
ctd_df["outlet"] = ctd_split.apply(lambda x: x[1]).astype("Int64")

# --- Apply to PLE dataset ---
ple_split = ple_df["IDOutlet"].apply(split_idoutlet)
ple_df["@logStream"] = ple_split.apply(lambda x: x[0])
ple_df["outlet"] = ple_split.apply(lambda x: x[1]).astype("Int64")


# Verify cableid consistency (last digit-CTD)
if "cableid" in ctd_df.columns:
    ctd_df["cableid_outlet"] = ctd_df["cableid"].apply(lambda x: int(str(x)[-1]) if pd.notna(x) and str(x)[-1].isdigit() else pd.NA).astype("Int64")
    mismatches = ctd_df[ctd_df["outlet"] != ctd_df["cableid_outlet"]]
    if not mismatches.empty:
        print("⚠️ Mismatches found between IDOutlet and cableid:")
        display(mismatches.head(10))
    else:
        print("✅ All CTD rows consistent: IDOutlet and cableid match.")

# --- Apply to SeccSessionStop dataset ---
sess_split = sess_df["IDOutlet"].apply(split_idoutlet)
sess_df["@logStream"] = sess_split.apply(lambda x: x[0])
sess_df["outlet"] = sess_split.apply(lambda x: x[1]).astype("Int64")

print("✅ @logStream + outlet columns created for CTD and SeccSessionStop.")
print("CTD sample:", ctd_df[["@logStream", "outlet"]].head())
print("PLE sample:", ple_df[["@logStream", "outlet"]].head())
print("Sessions sample:", sess_df[["@logStream", "outlet"]].head())


✅ All CTD rows consistent: IDOutlet and cableid match.
✅ @logStream + outlet columns created for CTD and SeccSessionStop.
CTD sample:   @logStream  outlet
0     yk59l5       1
1     0WO7nc       1
2     HfJX29       1
3     NWYmy6       1
4     mrPnq1       2
PLE sample:   @logStream  outlet
0     bM0EQk       2
1     bM0EQk       2
2     x4Uawq       2
3     x4Uawq       2
4     jERN9L       1
Sessions sample:   @logStream  outlet
0     Ubbd7a       2
1     bPeYpZ       2
2     64n1W8       2
3     FDL7ZG       2
4     U9Dn7E       1


In [6]:
for df in [ctd_df, sess_df, ple_df]:
    df["@logStream"] = df["@logStream"].astype(str).str.strip()


#### 7) Add day granularity
- All datasets now have a `day` column. Day-level aggregation reduces noise and aligns signals

In [7]:
#7 Add day column
for df in [ctd_df, sess_df, ple_df]:
    df["@timestamp"] = pd.to_datetime(df["@timestamp"], errors='coerce')
    df["day"] = df["@timestamp"].dt.floor("D")

#### 8) Safety: sort timestamps
**Purpose:** Sort by `@timestamp` before merge. 

In [8]:
# After loading and timestamp conversion:
for df in [ctd_df, sess_df, ple_df]:
    df["@timestamp"] = pd.to_datetime(df["@timestamp"], errors='coerce')
    df.sort_values(by="@timestamp", inplace=True)
    df["day"] = df["@timestamp"].dt.floor("D")

- 09) CTD aggregation: Purpose:** Aggregate CTD by outlet-day.  
- 10) CLG aggregation
- 11) PLE aggregation
- 12) Session aggregation

In [9]:
#9
ctd_summary = ctd_df.groupby(["@logStream", "outlet", "day"]).agg(
    CTD_count=("@timestamp", "count"),
    CTD_diff_mean=("diff", "mean"),
    CTD_factor_mean=("factor", "mean"),
    CTD_current_mean=("nowcur", "mean")
).reset_index()

In [10]:
#11
ple_summary = ple_df.groupby(["@logStream", "outlet", "day"]).agg(
    PLE_count=("@timestamp", "count")
).reset_index()

In [11]:
#12 Re-aggregate Sessions with New features(Duration)
sess_summary = sess_df.groupby(["@logStream", "outlet", "day"]).agg(
    Sess_count=("@timestamp", "count"),
    Sess_energy_mean=("energy", "mean"),
    Sess_temp_diff_mean=("diff", "mean"),
    # NEW:
    Sess_duration_mean=("duration", "mean"),   # avg duration per day (e.g., seconds or minutes)
    Sess_duration_total=("duration", "sum")    # total duration per day
).reset_index()

#### 13) Build master logstream-outlet-timeline
**Purpose:** Merge CTD, CLG, PLE, Sessions on logstream-outlet-day.  

In [12]:
master_timeline = (
    ctd_summary.merge(sess_summary, on=["@logStream", "outlet", "day"], how="outer")
               .merge(ple_summary, on=["@logStream", "outlet", "day"], how="outer")
)

# Preview
print("✅ Outlet-normalized timeline built!")
print(master_timeline.shape)
display(master_timeline.head())

✅ Outlet-normalized timeline built!
(658001, 13)


,@logStream,outlet,day,CTD_count,CTD_diff_mean,CTD_factor_mean,CTD_current_mean,Sess_count,Sess_energy_mean,Sess_temp_diff_mean,Sess_duration_mean,Sess_duration_total,PLE_count
0,003HJH,1,2025-04-04 00:00:00+00:00,NaN,NaN,NaN,NaN,1.0,171053.0,0.7774,2736.0,2736.0,NaN
1,003HJH,1,2025-04-18 00:00:00+00:00,NaN,NaN,NaN,NaN,1.0,21867.0,0.0000,2931.0,2931.0,NaN
2,003HJH,1,2025-04-29 00:00:00+00:00,NaN,NaN,NaN,NaN,1.0,46596.0,0.7774,3559.0,3559.0,NaN
3,003HJH,1,2025-04-30 00:00:00+00:00,NaN,NaN,NaN,NaN,1.0,41035.0,0.0000,1422.0,1422.0,NaN
4,003HJH,1,2025-05-04 00:00:00+00:00,NaN,NaN,NaN,NaN,1.0,14281.0,0.0000,1084.0,1084.0,NaN


- clean_master_timeline

- Counts (*_count) → fill missing with 0 (no event that day).

- Means (*_mean) → keep NaN (no measurement; don’t invent zeros).

- Make sure numeric columns are really numeric.

In [13]:
def clean_master_timeline(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # --- 1) Identify feature columns by suffix
    count_cols = [c for c in df.columns if c.endswith("_count")]
    mean_cols  = [c for c in df.columns if c.endswith("_mean")]

    # --- 2) Force numeric on feature columns (bad strings -> NaN)
    for c in count_cols + mean_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # --- 3) Fill counts with 0 (no event on that day)
    df[count_cols] = df[count_cols].fillna(0).astype("float64")

    # --- 4) Mask *means* by their family’s count (0 or NaN -> mean must be NaN)
    families = {
        "CTD": ["CTD_diff_mean", "CTD_factor_mean", "CTD_current_mean"],
        # "CLG": ["CLG_diff_mean", "CLG_temp1_mean", "CLG_temp2_mean", "CLG_current_mean"],
        # PLE has only counts
    }

    for fam, cols in families.items():
        cnt = f"{fam}_count"
        if cnt in df.columns:
            mask = (df[cnt].isna()) | (df[cnt] == 0)
            for col in cols:
                if col in df.columns:
                    # Any day with count==0/NaN cannot have a valid mean -> set to NaN
                    df.loc[mask, col] = np.nan

    # --- 5) Ensure key columns typed correctly
    if "@timestamp" in df.columns:
        df["@timestamp"] = pd.to_datetime(df["@timestamp"], errors="coerce")
    if "day" in df.columns:
        df["day"] = pd.to_datetime(df["day"], errors="coerce")
    if "outlet" in df.columns:
        df["outlet"] = pd.to_numeric(df["outlet"], errors="coerce").astype("Int64")

    return df

# 👉 apply right after the outer merges
master_timeline = clean_master_timeline(master_timeline)

print("✅ Cleaned + masked timeline.")
print("Counts (zero-filled):", [c for c in master_timeline.columns if c.endswith("_count")])
print("Means (masked by counts):", [c for c in master_timeline.columns if c.endswith("_mean")])

✅ Cleaned + masked timeline.
Counts (zero-filled): ['CTD_count', 'Sess_count', 'PLE_count']
Means (masked by counts): ['CTD_diff_mean', 'CTD_factor_mean', 'CTD_current_mean', 'Sess_energy_mean', 'Sess_temp_diff_mean', 'Sess_duration_mean']


- convert ms -> minutes BEFORE grouping


In [14]:
#convert ms -> minutes BEFORE grouping
sess_df["duration"] = pd.to_numeric(sess_df["duration"], errors="coerce")
sess_df["duration_min"] = sess_df["duration"] / 60000.0  # ms to minutes
# then aggregate on "duration_min" instead of "duration"

In [15]:
print(master_timeline.shape)
display(master_timeline.head(400))

(658001, 13)


,@logStream,outlet,day,CTD_count,CTD_diff_mean,CTD_factor_mean,CTD_current_mean,Sess_count,Sess_energy_mean,Sess_temp_diff_mean,Sess_duration_mean,Sess_duration_total,PLE_count
0,003HJH,1,2025-04-04 00:00:00+00:00,0.0,NaN,NaN,NaN,1.0,171053.000000,0.777400,2736.000000,2736.0,0.0
1,003HJH,1,2025-04-18 00:00:00+00:00,0.0,NaN,NaN,NaN,1.0,21867.000000,0.000000,2931.000000,2931.0,0.0
2,003HJH,1,2025-04-29 00:00:00+00:00,0.0,NaN,NaN,NaN,1.0,46596.000000,0.777400,3559.000000,3559.0,0.0
3,003HJH,1,2025-04-30 00:00:00+00:00,0.0,NaN,NaN,NaN,1.0,41035.000000,0.000000,1422.000000,1422.0,0.0
4,003HJH,1,2025-05-04 00:00:00+00:00,0.0,NaN,NaN,NaN,1.0,14281.000000,0.000000,1084.000000,1084.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,00No3j,2,2025-08-28 00:00:00+00:00,7.0,10.028571,0.136471,382.857143,5.0,31428.200000,4.457100,1701.800000,8509.0,0.0
396,00No3j,2,2025-08-29 00:00:00+00:00,2.0,5.050000,0.156550,320.000000,5.0,31125.400000,2.073080,2036.200000,10181.0,0.0
397,00No3j,2,2025-08-30 00:00:00+00:00,7.0,5.771429,0.146043,337.428571,10.0,32807.300000,2.280380,1600.400000,16004.0,0.0
398,00No3j,2,2025-08-31 00:00:00+00:00,4.0,7.500000,0.176525,309.750000,7.0,33271.428571,2.221157,1671.571429,11701.0,0.0


#### 14) Completeness check
**Purpose:** Count NaNs per feature.  

In [16]:
# 📝 List of columns to check
columns_to_check = [
   "CTD_count","Sess_count", "PLE_count"
]

# 🔍 Count zeros in each
zero_counts = (master_timeline[columns_to_check] == 0).sum()

# 🔢 Also show total number of rows for reference
total_rows = len(master_timeline)

# 🖨️ Print report
print(f"Total rows: {total_rows}\n")
for col in columns_to_check:
    count = int(zero_counts[col])
    pct = round(100 * count / total_rows, 2) if total_rows else 0
    if count == total_rows:
        print(f"⚠️ Column '{col}' has ALL {count} zeros ❗")
    else:
        print(f"✅ Column '{col}' has {count} zeros ({pct}%)")


Total rows: 658001

✅ Column 'CTD_count' has 447497 zeros (68.01%)
✅ Column 'Sess_count' has 24713 zeros (3.76%)
✅ Column 'PLE_count' has 643785 zeros (97.84%)


##### Filter the Outlets based on those have Issues
-  === Step 1:check Master_file_Outlet_issues for fileting the outlets
-  === Step 2: Clean filter file so it matches main dataset structure


In [17]:
filter_path = "../data_all/Master_file_Outlet_issues.xlsx"
filter_df = pd.read_excel(filter_path)

# Show schema
print("=== Filter Excel File ===")
print("🧾 Columns:", filter_df.columns.tolist())
print("\n📊 Dtypes:")
print(filter_df.dtypes)

print("\n Sample rows:")
display(filter_df.head())
print(filter_df.count())

#Rename columns to align with CTD / Sessions /  PLE
filter_df = filter_df.rename(columns={
    "ID": "@logStream",
    "Outlet": "outlet"
})

# Ensure outlet is integer type
filter_df["outlet"] = filter_df["outlet"].astype("Int64")

print("✅ Filter file cleaned.")
print(filter_df.head())


=== Filter Excel File ===
🧾 Columns: ['Outlet ID', 'ID', 'Outlet', 'Detection Date']

📊 Dtypes:
Outlet ID         object
ID                object
Outlet             int64
Detection Date    object
dtype: object

 Sample rows:


,Outlet ID,ID,Outlet,Detection Date
0,0AQbZI1,0AQbZI,1,2025-08-19 13:52
1,0PrWyp3,0PrWyp,3,2025-08-15 07:55
2,17xsOj2,17xsOj,2,2025-08-20 01:04
3,198r072,198r07,2,2025-08-03 15:37
4,198r074,198r07,4,2025-08-07 16:03


Outlet ID         253
ID                253
Outlet            253
Detection Date    253
dtype: int64
✅ Filter file cleaned.
  Outlet ID @logStream  outlet    Detection Date
0   0AQbZI1     0AQbZI       1  2025-08-19 13:52
1   0PrWyp3     0PrWyp       3  2025-08-15 07:55
2   17xsOj2     17xsOj       2  2025-08-20 01:04
3   198r072     198r07       2  2025-08-03 15:37
4   198r074     198r07       4  2025-08-07 16:03


##### Filtering + Aggregation + Master Timeline (correct order)


- Filtering 


In [ ]:
# === Normalize charger IDs everywhere first ===
for df in [ctd_df, sess_df, ple_df]:
    df["@logStream"] = df["@logStream"].astype(str).str.strip().str.lower()

filter_df["@logStream"] = filter_df["@logStream"].astype(str).str.strip().str.lower()

# # === Apply or skip filtering based on toggle ===
# apply_filter = False   # 🔘 Set to True to use the Excel filter, False to include ALL chargers

if apply_filter:
    # --- Use the filtered pairs from Excel file ---
    valid_pairs = set(zip(filter_df["@logStream"], filter_df["outlet"]))
    print(f"✅ Loaded {len(valid_pairs)} valid charger–outlet pairs from filter file.")

    ctd_filtered = ctd_df[ctd_df[["@logStream", "outlet"]]
                          .apply(tuple, axis=1)
                          .isin(valid_pairs)].copy()
    sess_filtered = sess_df[sess_df[["@logStream", "outlet"]]
                            .apply(tuple, axis=1)
                            .isin(valid_pairs)].copy()
    ple_filtered = ple_df[ple_df[["@logStream", "outlet"]]
                            .apply(tuple, axis=1)
                            .isin(valid_pairs)].copy()

else:
    # --- Skip filtering: use full datasets directly ---
    print("⚠️ Skipping Excel-based filtering. Using ALL chargers/outlets instead.")
    ctd_filtered = ctd_df.copy()
    sess_filtered = sess_df.copy()
    ple_filtered = ple_df.copy()

print("✅ Filtering complete.")
print("CTD rows before:", len(ctd_df), "→ after:", len(ctd_filtered))
print("Sessions rows before:", len(sess_df), "→ after:", len(sess_filtered))
print("PLE rows before:", len(ple_df), "→ after:", len(ple_filtered))


⚠️ Skipping Excel-based filtering. Using ALL chargers/outlets instead.
✅ Filtering complete.
CTD rows before: 718527 → after: 718527
Sessions rows before: 4215137 → after: 4215137
PLE rows before: 459943 → after: 459943


In [19]:
print("Unique outlets in Sessions:", sess_filtered["@logStream"].unique())


Unique outlets in Sessions: ['ubbd7a' 'bpeypz' '64n1w8' ... 'nfepdc' 'jcyipr' 'cqbugr']


- Aggregations

In [20]:
# === CTD daily summary ===
ctd_summary = ctd_filtered.groupby(["@logStream", "outlet", "day"]).agg(
    CTD_count=("@timestamp", "count"),
    CTD_diff_mean=("diff", "mean"),
    CTD_factor_mean=("factor", "mean"),
    CTD_current_mean=("nowcur", "mean")
).reset_index()

# === Session daily summary ===
sess_summary = sess_filtered.groupby(["@logStream", "outlet", "day"]).agg(
    Sess_count=("@timestamp", "count"),
    Sess_energy_mean=("energy", "mean"),
    Sess_temp_diff_mean=("diff", "mean"),
    Sess_duration_mean=("duration", "mean"),
    Sess_duration_total=("duration", "sum")
).reset_index()

# === PLE daily summary ===
ple_summary = ple_filtered.groupby(["@logStream", "outlet", "day"]).agg(
    PLE_count=("@timestamp", "count")
).reset_index()

- Merge into Master Timeline

In [21]:
# Merge all three summaries
master_timeline = (
    ctd_summary
    .merge(sess_summary, on=["@logStream", "outlet", "day"], how="outer")
    .merge(ple_summary, on=["@logStream", "outlet", "day"], how="outer")
)

# Sort + clean
master_timeline["day"] = pd.to_datetime(master_timeline["day"], errors="coerce")
master_timeline = master_timeline.sort_values(["@logStream", "outlet", "day"])


### Feature Engineering

In [22]:
def add_features(df):
    df = df.copy()
    # Energy/session
    df["Energy_per_session"] = np.where(
        df["Sess_count"] > 0,
        df["Sess_energy_mean"] / df["Sess_count"],
        np.nan
    )
    # Energy/duration
    df["Energy_per_duration"] = np.where(
        df["Sess_duration_mean"] > 0,
        df["Sess_energy_mean"] / df["Sess_duration_mean"],
        np.nan
    )
    # Ratios
    df["CTD_per_session"] = np.where(
        df["Sess_count"] > 0, df["CTD_count"] / df["Sess_count"], np.nan
    )
    df["PLE_per_session"] = np.where(
        df["Sess_count"] > 0, df["PLE_count"] / df["Sess_count"], np.nan
    )
    return df

outlet_timeline = add_features(master_timeline)


### Step 4 – Scoring

In [23]:
def score_outlets_combined(outlet_timeline, rolling_window=60, min_sessions=30):
    results = []
    for (charger, outlet), df in outlet_timeline.groupby(["@logStream", "outlet"]):
        if df["Sess_count"].sum() < min_sessions:
            continue

        temp = df["Sess_temp_diff_mean"].fillna(0)
        temp_med = temp.tail(90).median()
        temp_slope = np.gradient(temp.values) if len(temp) >= 3 else [0]

        sustained_rise = (temp_slope[-90:] > 0).mean() > 0.6 if len(temp) >= 90 else False
        sharp_rise_month = temp.pct_change(30).dropna().gt(0.8).any()
        sharp_rise_week = temp.pct_change(7).dropna().gt(2.0).any()

        # --- Rule contributions ---
        contributions = {}
        temp_score = 0

        if temp_med > 5:
            contributions["Median >5°C"] = 3
            temp_score += 3
        if temp_med > 10:
            contributions["Median >10°C"] = 2
            temp_score += 2
        if sustained_rise:
            contributions["Sustained rise (90d)"] = 3
            temp_score += 3
        if sharp_rise_month:
            contributions["80% rise in 30d"] = 3
            temp_score += 3
        if sharp_rise_week:
            contributions["200% rise in 7d"] = 3
            temp_score += 3

        stability_score = 1 if df["Sess_count"].sum() > 1000 else 0
        if stability_score:
            contributions["Stable session support"] = 1

        final_score = temp_score + stability_score

        results.append({
            "@logStream": charger,
            "outlet": outlet,
            "Final_score": final_score,
            "Contributions": contributions
        })

    ranking_df = pd.DataFrame(results).sort_values("Final_score", ascending=False).reset_index(drop=True)
    ranking_df["Rank"] = ranking_df.index + 1
    return ranking_df


### Step 5 – Plotting

In [24]:
# === Trend helper (with rise/fall markers) ===
def _add_trends(ax, x_dates, y, rolling_window=21, base_color="C0", label="Series"):
    """
    Plot raw series and its rolling mean trend on the given axis.
    Adds markers when a sustained upward or downward trend starts.
    """
    ax.plot(x_dates, y, marker='o', linestyle='-', color=base_color, alpha=0.5, label=label)

    y_roll = pd.Series(y, index=pd.to_datetime(x_dates)).rolling(
        window=rolling_window, min_periods=1
    ).mean()

    ax.plot(x_dates, y_roll.values, linestyle='--', linewidth=2,
            color="black", label=f"Trend ({rolling_window}d rolling)")

    # Skip rise/fall markers if too few points
    if len(y_roll.dropna()) < 5:
        return

    sign = np.sign(y_roll.diff().fillna(0).values)
    for i in range(len(sign) - 3):
        if all(sign[i:i+3] > 0):  # upward
            ax.annotate("↑ rise", (x_dates.iloc[i], y_roll.iloc[i]),
                        xytext=(0, 10), textcoords="offset points",
                        color=base_color, fontsize=9,
                        arrowprops=dict(arrowstyle="->", color=base_color))
            break


# === Outlier capping helper ===
def cap_outliers(series, upper_quantile=0.99):
    s = pd.to_numeric(series, errors="coerce")
    if s.dropna().empty:
        return s
    cap_value = s.quantile(upper_quantile)
    return np.minimum(s, cap_value)


# === Updated export function with score & reasons in title ===
def export_all_outlets_with_trends(outlet_timeline,
                                   ranking_df=None,
                                   output_dir="../plots_alloutlet_timelines_Grouped",
                                   rolling_window=30,
                                   duration_unit_label="min"):
    import os
    os.makedirs(output_dir, exist_ok=True)

    for (charger, outlet), df in outlet_timeline.groupby(["@logStream", "outlet"]):
        if df.empty:
            continue

        df = df.sort_values("day").copy()
        df["day"] = pd.to_datetime(df["day"], errors="coerce")

        # --- Apply outlier capping ---
        features_to_cap = {
            "Energy_per_session": 0.90,
            "Energy_per_duration": 0.90,
            "Sess_temp_diff_mean": 0.90,
            "Sess_duration_mean": 0.90,
            "CTD_count": 0.99,
            "PLE_count": 0.99,
            "CTD_per_session": 0.90,
            "PLE_per_session": 0.90,
        }
        for col, q in features_to_cap.items():
            if col in df:
                df[col] = cap_outliers(df[col], q)

        # --- Lookup score, rank, and reasons ---
        score_str, rank_str, reasons_str = "", "", ""
        if ranking_df is not None:
            row = ranking_df[(ranking_df["@logStream"] == charger) &
                             (ranking_df["outlet"] == outlet)]
            if not row.empty:
                score = row["Final_score"].values[0]
                rank = row["Rank"].values[0] if "Rank" in row.columns else row.index[0] + 1
                score_str = f" | Score: {score:.1f}"
                rank_str = f" | Rank: {rank}"
                if "Reasons" in row.columns:
                    reasons_str = f"\nReasons: {row['Reasons'].values[0]}"

        # --- Figure setup ---
        fig, axs = plt.subplots(5, 1, figsize=(18, 20), sharex=True)
        fig.suptitle(f"{charger} – Outlet {outlet}{score_str}{rank_str}{reasons_str}",
                     fontsize=14)

        # 1) Energy Usage (Wh/session + Wh/min)
        if "Energy_per_session" in df:
            _add_trends(axs[0], df["day"], df["Energy_per_session"].to_numpy(dtype=float),
                        rolling_window, base_color="green", label="Wh/Session")
        if "Energy_per_duration" in df:
            _add_trends(axs[0], df["day"], df["Energy_per_duration"].to_numpy(dtype=float),
                        rolling_window, base_color="darkgreen", label=f"Wh/{duration_unit_label}")
        axs[0].set_title("Energy Usage")
        axs[0].set_ylabel("Wh"); axs[0].legend(loc="upper left")

        # 2) Session Temperature Diff
        if "Sess_temp_diff_mean" in df:
            _add_trends(axs[1], df["day"], df["Sess_temp_diff_mean"].to_numpy(dtype=float),
                        rolling_window, base_color="orange", label="Temp Diff (°C)")
        axs[1].set_title("Session Temperature Diff (°C)")
        axs[1].set_ylabel("°C"); axs[1].legend(loc="upper left")

        # 3) Counts (CTD + PLE)
        if "CTD_count" in df:
            _add_trends(axs[2], df["day"], df["CTD_count"].to_numpy(dtype=float),
                        rolling_window, base_color="brown", label="CTD Count")
        if "PLE_count" in df:
            _add_trends(axs[2], df["day"], df["PLE_count"].to_numpy(dtype=float),
                        rolling_window, base_color="blue", label="PLE Count")
        axs[2].set_title("Event Counts")
        axs[2].set_ylabel("count"); axs[2].legend(loc="upper left")

        # 4) Ratios (CTD/session + PLE/session)
        if "CTD_per_session" in df:
            _add_trends(axs[3], df["day"], df["CTD_per_session"].to_numpy(dtype=float),
                        rolling_window, base_color="red", label="CTD per Session")
        if "PLE_per_session" in df:
            _add_trends(axs[3], df["day"], df["PLE_per_session"].to_numpy(dtype=float),
                        rolling_window, base_color="blue", label="PLE per Session")
        axs[3].axhline(1, color="gray", linestyle="--", alpha=0.6)
        axs[3].set_title("Ratios (per Session)")
        axs[3].set_ylabel("ratio"); axs[3].legend(loc="upper left")

        # 5) Session Duration
        if "Sess_duration_mean" in df:
            _add_trends(axs[4], df["day"], df["Sess_duration_mean"].to_numpy(dtype=float),
                        rolling_window, base_color="purple", label=f"Duration ({duration_unit_label})")
        axs[4].set_title(f"Session Duration (mean, {duration_unit_label})")
        axs[4].set_ylabel(duration_unit_label); axs[4].legend(loc="upper left")

        # Format x-axis
        axs[4].xaxis.set_major_locator(mdates.MonthLocator())
        axs[4].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        plt.xticks(rotation=45)

        plt.tight_layout(rect=[0, 0.02, 1, 0.97])

        # --- Save file with rank in name if available ---
        if ranking_df is not None and not row.empty:
            filename = f"Rank{rank}_{charger}_Outlet{outlet}.png".replace("/", "_")
        else:
            filename = f"{charger}_Outlet{outlet}_grouped.png".replace("/", "_")

        plt.savefig(os.path.join(output_dir, filename), dpi=150)
        plt.close(fig)

    print(f"✅ Batch export done: grouped plots saved to {output_dir}")


### Step 6 – Export Top 20

In [25]:
def export_top_outlets(outlet_timeline, ranking_df, n=50, output_dir="../plots_top50"):
    import os
    os.makedirs(output_dir, exist_ok=True)

    # Take the top N from the ranking list
    top_list = ranking_df.head(n).copy()

    # Create set of charger–outlet pairs
    selected_pairs = set(zip(top_list["@logStream"], top_list["outlet"]))

    # Filter outlet_timeline to just those pairs
    filtered = outlet_timeline[outlet_timeline[["@logStream", "outlet"]]
                               .apply(tuple, axis=1)
                               .isin(selected_pairs)]

    # Reuse your export function
    export_all_outlets_with_trends(filtered,
                                   ranking_df=ranking_df,
                                   output_dir=output_dir,
                                   rolling_window=30,
                                   duration_unit_label="min")

    # --- Save the top list table ---
    table_path = os.path.join(output_dir, "Top20_table.xlsx")
    top_list.to_excel(table_path, index=False)

    print(f"✅ Exported plots + saved Top20 table to {table_path}")

    return top_list


    # 1. Build ranking
ranking_df = score_outlets_combined(outlet_timeline, rolling_window=60)

# 2. Export top 20 (plots + table)
top50 = export_top_outlets(outlet_timeline, ranking_df, n=50, output_dir="../plots_top50_281025")

# 3. Display table inline in notebook
display(top50)



✅ Batch export done: grouped plots saved to ../plots_top50_281025
✅ Exported plots + saved Top20 table to ../plots_top50_281025\Top20_table.xlsx


,@logStream,outlet,Final_score,Contributions,Rank
0,7uu7co,1,12,"{'Median >5°C': 3, 'Median >10°C': 2, '80% ris...",1
1,gkybxl,3,11,"{'Median >5°C': 3, 'Median >10°C': 2, '80% ris...",2
2,q4vkdh,3,11,"{'Median >5°C': 3, 'Median >10°C': 2, '80% ris...",3
3,cjuqjx,1,10,"{'Median >5°C': 3, '80% rise in 30d': 3, '200%...",4
4,h37rlm,1,10,"{'Median >5°C': 3, '80% rise in 30d': 3, '200%...",5
5,zhsxbz,2,10,"{'Median >5°C': 3, '80% rise in 30d': 3, '200%...",6
6,0aqbzi,1,10,"{'Median >5°C': 3, '80% rise in 30d': 3, '200%...",7
7,cjuqjx,2,10,"{'Median >5°C': 3, '80% rise in 30d': 3, '200%...",8
8,zkvb32,2,10,"{'Median >5°C': 3, '80% rise in 30d': 3, '200%...",9
9,rfhsw1,1,10,"{'Median >5°C': 3, '80% rise in 30d': 3, '200%...",10


In [26]:
# === Export top-N chargers in "csonf21|afnlo23|..." format ===
def export_top_charger_list(top_df, n=50):
    chargers = top_df.head(n)["@logStream"].unique()
    charger_str = "|".join(chargers)
    print("Pipe-separated charger list:")
    print(charger_str)
    return charger_str

# Usage (after you have top20 from export_top_outlets):
pipe_str = export_top_charger_list(top50, n=50)


Pipe-separated charger list:
7uu7co|gkybxl|q4vkdh|cjuqjx|h37rlm|zhsxbz|0aqbzi|zkvb32|rfhsw1|xwfc5g|obynih|9kxflp|8k6axi|k9oexg|j0vifw|cqkzw3|srvclh|kjpw49|a2zoag|ah0die|bpeypz|ywghbs|gat0ck|krijnx|klz8uc|nqymby|obibh9|olg0bp|tjtoil|zt9lsw|xihdgt|czrtl4|053wej|0a6iko|08rty5|07fhal|xdc81p|crqaxd|crxbxw|d664uu|d683kt|xctvp4|xdbymt


In [27]:
# === Export top-N charger-outlet pairs in "csonf21-1|csonf21-2|afnlo23-1|..." format ===
def export_top_charger_outlets(top_df, n=50):
    pairs = top_df.head(n)[["@logStream", "outlet"]].dropna()
    pairs_str = "|".join([f"{row['@logStream']}-{int(row['outlet'])}" for _, row in pairs.iterrows()])
    print("Pipe-separated charger-outlet list:")
    print(pairs_str)
    return pairs_str

# Usage (after you have top20 from export_top_outlets):
pipe_str = export_top_charger_outlets(top50, n=50)


Pipe-separated charger-outlet list:
7uu7co-1|gkybxl-3|q4vkdh-3|cjuqjx-1|h37rlm-1|zhsxbz-2|0aqbzi-1|cjuqjx-2|zkvb32-2|rfhsw1-1|xwfc5g-2|obynih-1|9kxflp-3|8k6axi-2|k9oexg-2|j0vifw-2|cqkzw3-1|srvclh-2|kjpw49-1|a2zoag-1|ah0die-2|bpeypz-2|ywghbs-1|gat0ck-2|krijnx-1|klz8uc-3|nqymby-1|obibh9-2|olg0bp-1|tjtoil-1|zt9lsw-2|xihdgt-1|czrtl4-2|053wej-1|0a6iko-2|0a6iko-1|08rty5-2|08rty5-1|07fhal-3|053wej-2|xdc81p-2|crqaxd-1|crqaxd-2|crxbxw-1|crxbxw-2|d664uu-1|d664uu-2|d683kt-1|xctvp4-3|xdbymt-1
